# DocLite: Knowledge Distillation for Document Understanding

**Teacher:** LayoutLMv3 → **Student:** LiLT (distilled)

Experiments: baselines + distillation on FUNSD and SROIE

In [ ]:
# Setup
!rm -rf doclite
!git clone https://github.com/lukeengel/doclite.git
%cd doclite
!pip install -r requirements.txt -q
!apt-get install -y tesseract-ocr -qq

import torch, gc
print(f"CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# Uncomment if you have SROIE on Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !unzip -qo /content/drive/MyDrive/sroie.zip -d data/

In [ ]:
import torch, gc
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, LayoutLMv3Config, LayoutLMv3ForTokenClassification
from collections import Counter

from doclite.configs.core import ENV, WEIGHTS
from doclite.data.document_dataset import DocumentDataset
from doclite.data.collate import collate_document_batch
from doclite.eval.evaluate import evaluate_student
from doclite.models.teacher import TeacherModel
from doclite.models.student import StudentModel
from doclite.distill.distill_loss import compute_distill_loss
from build_funsd_examples import load_funsd_split

device = "cuda"
all_results = {}

NUM_EPOCHS = 10
LR = 5e-5

class LayoutLMv3EvalWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, input_ids, attention_mask, bbox, **kwargs):
        fwd = dict(input_ids=input_ids, attention_mask=attention_mask, bbox=bbox)
        if 'pixel_values' in kwargs:
            fwd['pixel_values'] = kwargs['pixel_values']
        return {"logits": self.model(**fwd).logits}

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU: {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total")

print("Ready")

---
# FUNSD
---

In [ ]:
FUNSD_ROOT = ENV.DATA / "funsd"
tokenizer = AutoTokenizer.from_pretrained("microsoft/layoutlmv3-base")

funsd_train = load_funsd_split(FUNSD_ROOT / "training_data" / "annotations", tokenizer)
funsd_test = load_funsd_split(FUNSD_ROOT / "testing_data" / "annotations", tokenizer)

print(f"Train: {len(funsd_train)} | Test: {len(funsd_test)}")

### FUNSD: LayoutLMv3 Baseline (Teacher)

In [ ]:
BS = 8
train_loader = DataLoader(DocumentDataset(funsd_train), batch_size=BS, shuffle=True, collate_fn=collate_document_batch)
test_loader = DataLoader(DocumentDataset(funsd_test), batch_size=BS, shuffle=False, collate_fn=collate_document_batch)

config = LayoutLMv3Config.from_pretrained("microsoft/layoutlmv3-base", num_labels=4, output_hidden_states=True, output_attentions=True)
model = LayoutLMv3ForTokenClassification.from_pretrained("microsoft/layoutlmv3-base", config=config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
wrapper = LayoutLMv3EvalWrapper(model).to(device)

all_results["teacher_params"] = sum(p.numel() for p in model.parameters())

results = []
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], bbox=batch["bbox"], labels=batch["labels"])
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
    metrics = evaluate_student(wrapper, test_loader, device=device)
    results.append(metrics)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | loss={total_loss/len(train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

all_results["funsd_teacher"] = max(results, key=lambda x: x['token_f1'])
print(f"\n>>> Best F1: {all_results['funsd_teacher']['token_f1']:.4f}")

del model, optimizer, wrapper
free_gpu()

### FUNSD: LiLT Baseline (Student)

In [ ]:
model = StudentModel("SCUT-DLVCLab/lilt-roberta-en-base", num_labels=4).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

all_results["student_params"] = sum(p.numel() for p in model.parameters())

results = []
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model.model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], bbox=batch["bbox"], labels=batch["labels"])
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
    metrics = evaluate_student(model, test_loader, device=device)
    results.append(metrics)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | loss={total_loss/len(train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

all_results["funsd_student"] = max(results, key=lambda x: x['token_f1'])
print(f"\n>>> Best F1: {all_results['funsd_student']['token_f1']:.4f}")

del model, optimizer
free_gpu()

### FUNSD: DocLite Distillation

Teacher in **float16** (frozen, saves ~half GPU memory) + student in float32.

In [ ]:
BS_DISTILL = 4
train_loader = DataLoader(DocumentDataset(funsd_train), batch_size=BS_DISTILL, shuffle=True, collate_fn=collate_document_batch)
test_loader = DataLoader(DocumentDataset(funsd_test), batch_size=BS_DISTILL, shuffle=False, collate_fn=collate_document_batch)

# Teacher in float16 to save memory (it's frozen, no precision needed for gradients)
teacher = TeacherModel("microsoft/layoutlmv3-base", num_labels=4).half().to(device)
for param in teacher.parameters():
    param.requires_grad = False

student = StudentModel("SCUT-DLVCLab/lilt-roberta-en-base", num_labels=4).to(device)
optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.01)

free_gpu()

results = []
for epoch in range(NUM_EPOCHS):
    teacher.eval()
    student.train()
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()

        model_inputs = {"input_ids": batch["input_ids"], "attention_mask": batch["attention_mask"], "bbox": batch["bbox"]}

        with torch.amp.autocast('cuda'):
            teacher_out = teacher(**model_inputs)

        # Cast teacher outputs to float32 for loss computation
        teacher_out = {k: v.float() if torch.is_tensor(v) else tuple(t.float() for t in v) for k, v in teacher_out.items()}

        student_hf_out = student.model(
            input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
            bbox=batch["bbox"], labels=batch["labels"],
            output_hidden_states=True, output_attentions=True, return_dict=True,
        )
        student_out = {"logits": student_hf_out.logits, "hidden_states": student_hf_out.hidden_states, "attentions": student_hf_out.attentions}

        distill_losses = compute_distill_loss(teacher_out, student_out, attention_mask=batch["attention_mask"])
        loss = distill_losses["loss_total"] + WEIGHTS.DELTA_TASK * student_hf_out.loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        if step % 10 == 0:
            print(f"  Epoch {epoch+1} | Step {step}/{len(train_loader)} | loss={loss.item():.4f}")

    metrics = evaluate_student(student, test_loader, device=device)
    results.append(metrics)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | loss={total_loss/len(train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

all_results["funsd_distill"] = max(results, key=lambda x: x['token_f1'])
print(f"\n>>> Best F1: {all_results['funsd_distill']['token_f1']:.4f}")

del teacher, student, optimizer
free_gpu()

### FUNSD Results

In [ ]:
tp = all_results["teacher_params"]
sp = all_results["student_params"]
ft = all_results["funsd_teacher"]
fs = all_results["funsd_student"]
fd = all_results["funsd_distill"]

print("=" * 75)
print("FUNSD RESULTS")
print("=" * 75)
print(f"{'Model':<30s} {'Acc':>8s} {'F1 macro':>10s} {'F1 micro':>10s} {'Params':>12s}")
print("-" * 75)
print(f"{'LayoutLMv3 (teacher)':<30s} {ft['token_acc']:>8.4f} {ft['token_f1_macro']:>10.4f} {ft['token_f1_micro']:>10.4f} {tp:>12,}")
print(f"{'LiLT (baseline)':<30s} {fs['token_acc']:>8.4f} {fs['token_f1_macro']:>10.4f} {fs['token_f1_micro']:>10.4f} {sp:>12,}")
print(f"{'LiLT + DocLite (ours)':<30s} {fd['token_acc']:>8.4f} {fd['token_f1_macro']:>10.4f} {fd['token_f1_micro']:>10.4f} {sp:>12,}")
print("=" * 75)
gap = (fd['token_f1'] - fs['token_f1']) / (ft['token_f1'] - fs['token_f1'] + 1e-8) * 100
print(f"Distillation closed {gap:.1f}% of the teacher-student F1 gap")
print(f"Compression: {tp/sp:.1f}x fewer parameters")

---
# FUNSD: Multimodal Distillation (Teacher sees images)

Teacher sees **text + layout + image** → Student sees only **text + layout**

---

In [ ]:
TRAIN_IMG_DIR = FUNSD_ROOT / "training_data" / "images"
TEST_IMG_DIR = FUNSD_ROOT / "testing_data" / "images"

print("Loading data with images...")
funsd_train_img = load_funsd_split(FUNSD_ROOT / "training_data" / "annotations", tokenizer, image_dir=TRAIN_IMG_DIR)
funsd_test_img = load_funsd_split(FUNSD_ROOT / "testing_data" / "annotations", tokenizer, image_dir=TEST_IMG_DIR)
print(f"Done. Has pixel_values: {'pixel_values' in funsd_train_img[0]}")

### LayoutLMv3 + Image Baseline

In [ ]:
BS = 4  # smaller batch for image data (more memory per sample)
train_loader = DataLoader(DocumentDataset(funsd_train_img), batch_size=BS, shuffle=True, collate_fn=collate_document_batch)
test_loader = DataLoader(DocumentDataset(funsd_test_img), batch_size=BS, shuffle=False, collate_fn=collate_document_batch)

config = LayoutLMv3Config.from_pretrained("microsoft/layoutlmv3-base", num_labels=4, output_hidden_states=True, output_attentions=True)
model = LayoutLMv3ForTokenClassification.from_pretrained("microsoft/layoutlmv3-base", config=config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
wrapper = LayoutLMv3EvalWrapper(model).to(device)

results = []
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                    bbox=batch["bbox"], pixel_values=batch.get("pixel_values"), labels=batch["labels"])
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
    metrics = evaluate_student(wrapper, test_loader, device=device)
    results.append(metrics)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | loss={total_loss/len(train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

all_results["funsd_teacher_img"] = max(results, key=lambda x: x['token_f1'])
print(f"\n>>> Best F1: {all_results['funsd_teacher_img']['token_f1']:.4f}")

del model, optimizer, wrapper
free_gpu()

### Multimodal DocLite Distillation

In [ ]:
BS_DISTILL = 2
train_loader = DataLoader(DocumentDataset(funsd_train_img), batch_size=BS_DISTILL, shuffle=True, collate_fn=collate_document_batch)
test_loader = DataLoader(DocumentDataset(funsd_test_img), batch_size=BS_DISTILL, shuffle=False, collate_fn=collate_document_batch)

teacher = TeacherModel("microsoft/layoutlmv3-base", num_labels=4).half().to(device)
for param in teacher.parameters():
    param.requires_grad = False

student = StudentModel("SCUT-DLVCLab/lilt-roberta-en-base", num_labels=4).to(device)
optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.01)

free_gpu()

results = []
for epoch in range(NUM_EPOCHS):
    teacher.eval()
    student.train()
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()

        teacher_inputs = {"input_ids": batch["input_ids"], "attention_mask": batch["attention_mask"], "bbox": batch["bbox"]}
        if "pixel_values" in batch:
            teacher_inputs["pixel_values"] = batch["pixel_values"]

        with torch.amp.autocast('cuda'):
            teacher_out = teacher(**teacher_inputs)
        teacher_out = {k: v.float() if torch.is_tensor(v) else tuple(t.float() for t in v) for k, v in teacher_out.items()}

        student_hf_out = student.model(
            input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
            bbox=batch["bbox"], labels=batch["labels"],
            output_hidden_states=True, output_attentions=True, return_dict=True,
        )
        student_out = {"logits": student_hf_out.logits, "hidden_states": student_hf_out.hidden_states, "attentions": student_hf_out.attentions}

        distill_losses = compute_distill_loss(teacher_out, student_out, attention_mask=batch["attention_mask"])
        loss = distill_losses["loss_total"] + WEIGHTS.DELTA_TASK * student_hf_out.loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        if step % 10 == 0:
            print(f"  Epoch {epoch+1} | Step {step}/{len(train_loader)} | loss={loss.item():.4f}")

    metrics = evaluate_student(student, test_loader, device=device)
    results.append(metrics)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | loss={total_loss/len(train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

all_results["funsd_mm_distill"] = max(results, key=lambda x: x['token_f1'])
print(f"\n>>> Best F1: {all_results['funsd_mm_distill']['token_f1']:.4f}")

del teacher, student, optimizer
free_gpu()

---
# SROIE (Receipts)

Skip if SROIE not uploaded.

---

In [ ]:
from build_sroie_examples import load_sroie_split, NUM_LABELS as NUM_LABELS_SROIE

SROIE_ROOT = ENV.DATA / "sroie"
sroie_train = load_sroie_split(SROIE_ROOT/"train"/"box", SROIE_ROOT/"train"/"entities", tokenizer)
sroie_test = load_sroie_split(SROIE_ROOT/"test"/"box", SROIE_ROOT/"test"/"entities", tokenizer)
print(f"SROIE — Train: {len(sroie_train)} | Test: {len(sroie_test)}")

ID2LABEL_SROIE = {0: "company", 1: "date", 2: "address", 3: "total", 4: "other", -100: "[PAD]"}
all_labels_s = [l for ex in sroie_train for l in ex["labels"] if l != -100]
counts_s = Counter(all_labels_s)
print("\nLabel distribution:")
for lid, c in sorted(counts_s.items()):
    print(f"  {ID2LABEL_SROIE[lid]:>10s}: {c:>5d} ({c/len(all_labels_s)*100:.1f}%)")

### SROIE: Teacher Baseline

In [ ]:
BS = 8
train_loader = DataLoader(DocumentDataset(sroie_train), batch_size=BS, shuffle=True, collate_fn=collate_document_batch)
test_loader = DataLoader(DocumentDataset(sroie_test), batch_size=BS, shuffle=False, collate_fn=collate_document_batch)

config = LayoutLMv3Config.from_pretrained("microsoft/layoutlmv3-base", num_labels=NUM_LABELS_SROIE, output_hidden_states=True, output_attentions=True)
model = LayoutLMv3ForTokenClassification.from_pretrained("microsoft/layoutlmv3-base", config=config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
wrapper = LayoutLMv3EvalWrapper(model).to(device)

results = []
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], bbox=batch["bbox"], labels=batch["labels"])
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
    metrics = evaluate_student(wrapper, test_loader, device=device)
    results.append(metrics)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | loss={total_loss/len(train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

all_results["sroie_teacher"] = max(results, key=lambda x: x['token_f1'])
print(f"\n>>> Best F1: {all_results['sroie_teacher']['token_f1']:.4f}")

del model, optimizer, wrapper
free_gpu()

### SROIE: Student Baseline

In [ ]:
model = StudentModel("SCUT-DLVCLab/lilt-roberta-en-base", num_labels=NUM_LABELS_SROIE).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

results = []
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model.model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], bbox=batch["bbox"], labels=batch["labels"])
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
    metrics = evaluate_student(model, test_loader, device=device)
    results.append(metrics)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | loss={total_loss/len(train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

all_results["sroie_student"] = max(results, key=lambda x: x['token_f1'])
print(f"\n>>> Best F1: {all_results['sroie_student']['token_f1']:.4f}")

del model, optimizer
free_gpu()

### SROIE: DocLite Distillation

In [ ]:
BS_DISTILL = 4
train_loader = DataLoader(DocumentDataset(sroie_train), batch_size=BS_DISTILL, shuffle=True, collate_fn=collate_document_batch)
test_loader = DataLoader(DocumentDataset(sroie_test), batch_size=BS_DISTILL, shuffle=False, collate_fn=collate_document_batch)

teacher = TeacherModel("microsoft/layoutlmv3-base", num_labels=NUM_LABELS_SROIE).half().to(device)
for param in teacher.parameters():
    param.requires_grad = False

student = StudentModel("SCUT-DLVCLab/lilt-roberta-en-base", num_labels=NUM_LABELS_SROIE).to(device)
optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.01)

free_gpu()

results = []
for epoch in range(NUM_EPOCHS):
    teacher.eval()
    student.train()
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()

        model_inputs = {"input_ids": batch["input_ids"], "attention_mask": batch["attention_mask"], "bbox": batch["bbox"]}

        with torch.amp.autocast('cuda'):
            teacher_out = teacher(**model_inputs)
        teacher_out = {k: v.float() if torch.is_tensor(v) else tuple(t.float() for t in v) for k, v in teacher_out.items()}

        student_hf_out = student.model(
            input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
            bbox=batch["bbox"], labels=batch["labels"],
            output_hidden_states=True, output_attentions=True, return_dict=True,
        )
        student_out = {"logits": student_hf_out.logits, "hidden_states": student_hf_out.hidden_states, "attentions": student_hf_out.attentions}

        distill_losses = compute_distill_loss(teacher_out, student_out, attention_mask=batch["attention_mask"])
        loss = distill_losses["loss_total"] + WEIGHTS.DELTA_TASK * student_hf_out.loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    metrics = evaluate_student(student, test_loader, device=device)
    results.append(metrics)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | loss={total_loss/len(train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

all_results["sroie_distill"] = max(results, key=lambda x: x['token_f1'])
print(f"\n>>> Best F1: {all_results['sroie_distill']['token_f1']:.4f}")

del teacher, student, optimizer
free_gpu()

---
# Final Summary
---

In [ ]:
tp = all_results.get("teacher_params", 0)
sp = all_results.get("student_params", 0)

print("=" * 80)
print("COMPLETE RESULTS")
print("=" * 80)

def print_row(name, r, inputs, params=None):
    p = f"{params:>12,}" if params else f"{'':>12s}"
    micro = r.get('token_f1_micro', r.get('token_acc', 0))
    macro = r.get('token_f1_macro', r.get('token_f1', 0))
    print(f"{name:<35s} {inputs:<8s} {r['token_acc']:>6.4f} {macro:>8.4f} {micro:>8.4f} {p}")

print(f"\n{'FUNSD (Forms)':^80s}")
print(f"{'Model':<35s} {'Input':<8s} {'Acc':>6s} {'F1 mac':>8s} {'F1 mic':>8s} {'Params':>12s}")
print("-" * 80)
print_row("LayoutLMv3 (teacher)", all_results["funsd_teacher"], "T+L", tp)
print_row("LiLT (baseline)", all_results["funsd_student"], "T+L", sp)
print_row("LiLT + DocLite (ours)", all_results["funsd_distill"], "T+L", sp)
if "funsd_teacher_img" in all_results:
    print_row("LayoutLMv3 + Image", all_results["funsd_teacher_img"], "T+L+I", tp)
if "funsd_mm_distill" in all_results:
    print_row("LiLT + DocLite (multimodal)", all_results["funsd_mm_distill"], "T+L", sp)

if "sroie_teacher" in all_results:
    print(f"\n{'SROIE (Receipts)':^80s}")
    print(f"{'Model':<35s} {'Input':<8s} {'Acc':>6s} {'F1 mac':>8s} {'F1 mic':>8s}")
    print("-" * 80)
    print_row("LayoutLMv3 (teacher)", all_results["sroie_teacher"], "T+L")
    print_row("LiLT (baseline)", all_results["sroie_student"], "T+L")
    print_row("LiLT + DocLite (ours)", all_results["sroie_distill"], "T+L")

print("\n" + "=" * 80)
print(f"T=Text, L=Layout, I=Image | Compression: {tp/sp:.1f}x")
print("=" * 80)